# Module 5: FedAvg Baselines

This staged notebook owns the clean FedAvg and attacked FedAvg handoff artifacts for Module 5.


## Stage Goal

Run the same FedAvg server twice: once with no malicious clients and once with the configured Module 4 malicious-client recipe. Save both result JSON files plus update diagnostics from the attacked run.


## 1. Notebook Setup

Load the split-notebook helper layer and keep imports independent of whether Jupyter starts from the repository root or from `5_Defensive_FL/`.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
REPO_ROOT = MODULE_DIR.parent

for path in (REPO_ROOT, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_utils import (
    load_context,
    record_config_snapshot,
    run_fedavg_baselines,
    save_update_diagnostics,
    run_result_path,
    validate_artifacts,
    validate_result_collection,
)


## 2. Configuration and Artifact Setup

This stage uses `fedavg_baselines_config.yaml`. The evaluation loader uses the Module 4 `attack_eval` validation split so Module 5 defense metrics are measured on the same held-out subset as the attack notebook.


In [ ]:
context = load_context("fedavg_baselines_config.yaml", stage_name="fedavg_baselines")
config_snapshot_path = record_config_snapshot(context)
config_snapshot_path


## 3. Clean and Attacked FedAvg

The clean run sets malicious fraction and poison rate to zero. The attacked run uses the configured PGD malicious-client recipe.


In [ ]:
run_results = run_fedavg_baselines(context)
baseline_validation = validate_result_collection(
    context,
    run_results,
    required_runs=["clean_fedavg", "attacked_fedavg"],
)
validate_artifacts(run_result_path(context, name) for name in run_results)
baseline_validation


## 4. Update Diagnostics

Use the attacked FedAvg run to record client update norms, malicious-client flags, cosine-to-mean behavior, and distance to the coordinate-wise median.


In [ ]:
norm_rows = save_update_diagnostics(context, run_results["attacked_fedavg"])
validate_artifacts([
    context.artifact_path("module5_update_diagnostics.json"),
    context.artifact_path("module5_update_norms.png"),
])
norm_rows[:5]


## Handoff Artifacts

Run `defense_comparison.ipynb` after these files exist.


In [ ]:
handoff_artifacts = validate_artifacts([
    config_snapshot_path,
    context.artifact_path("module5_clean_fedavg.json"),
    context.artifact_path("module5_attacked_fedavg.json"),
    context.artifact_path("module5_update_diagnostics.json"),
    context.artifact_path("module5_update_norms.png"),
])
handoff_artifacts
